# Feature Engineering — RFM Analysis

In this notebook, we will convert order-level data into a **customer-level** dataset, 
so that each row represents a customer rather than an individual order or item.

We will create the following features:
- **Recency (R)**: How many days ago the customer placed their last order
- **Frequency (F)**: The total number of orders placed by the customer
- **Monetary (M)**: The total amount spent by the customer

Bonus features:
- **Avg Review Score**: An indicator of customer satisfaction
- **Avg Delivery Delay**: The average delay in the delivery of the customer's orders

These features will serve as the input for K-Means clustering later on.

## Step 1: Loading Data

We are loading the cleaned and merged dataset created in `02_Data_Preprocessing.ipynb`.
We have retained only **delivered** orders, as cancelled or unavailable orders do not reflect the customer's actual purchasing behavior.

In [1]:
import pandas as pd 

df = pd.read_csv("../data/intermediate/cleaned_orders.csv")

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp"
]
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df = df[df["order_status"]== "delivered"].copy()

print("Shape:", df.shape)
    

Shape: (115720, 35)


## Step 2: Calculating Recency

Recency = The number of days between "today" (or the dataset's last known date) and the customer's **last order date**.

Formula: `Recency = Reference Date - Customer's Last Order Date`

Instead of "today," we are using the dataset's **max order date** as the reference point because this is 
historical data (up to 2018); using the actual "today" would be incorrect—the Recency values ​​
would become artificially inflated and meaningless.

In [2]:
reference_date = df["order_purchase_timestamp"].max()
print("Reference date (max order date in data):", reference_date)

recency_df = df.groupby("customer_unique_id")["order_purchase_timestamp"].max().reset_index()
recency_df.columns = ["customer_unique_id", "last_order_date"]

recency_df["recency_days"] = (reference_date - recency_df["last_order_date"]).dt.days

print("\nRecency preview:")
recency_df.head()

Reference date (max order date in data): 2018-08-29 15:00:37

Recency preview:


,customer_unique_id,last_order_date,recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,111
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,114
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,536
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,320
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,287


## Step 3: Calculating Frequency

Frequency = The total number of **unique orders** placed by a customer.

We already observed during the EDA that 97% of customers place only a single order—
so, while this feature will have low variance, it is essential for clearly identifying the 3% who are repeat customers.

In [3]:
frequency_df = df.groupby("customer_unique_id")["order_id"].nunique().reset_index()
frequency_df .columns = ["customer_unique_id", "frequency"]

print("Frequency preview:")
frequency_df.head()

print("\nFrequency distribution:")
print(frequency_df["frequency"].value_counts().head())

Frequency preview:

Frequency distribution:
frequency
1    90556
2     2573
3      181
4       28
5        9
Name: count, dtype: int64


## Step 4: Calculating the Monetary Value

Monetary = The total amount a customer **spent** (across all orders).

We will use the `payment_value` column because it represents the actual amount paid by the customer 
(including price, freight, and any additional charges), making it a more accurate 
measure of monetary value than the `price` column.

In [4]:
monetary_df = df.groupby("customer_unique_id")["payment_value"].sum().reset_index()
monetary_df.columns = ["customer_unique_id", "monetary"]

print("Monetary preview:")
monetary_df.head()

print("\nMonetary status:")
print(monetary_df["monetary"].describe())

Monetary preview:

Monetary status:
count     93357.000000
mean        212.966838
std         646.226951
min           9.590000
25%          63.830000
50%         113.140000
75%         202.640000
max      109312.640000
Name: monetary, dtype: float64


## Important Business Insight

The monetary distribution is heavily right-skewed (mean ₹213 vs. median ₹113) and includes an extreme outlier (₹109,312 — approximately 967 times the median). This outlier could distort clustering results if `StandardScaler` is used directly. To address this, one should either cap the outlier (capping/winsorization) or use `RobustScaler`, which is less sensitive to outliers.

## Step 5: Bonus Features — Avg Review Score and Avg Delivery Time

In addition to RFM, we will create two additional behavioral features:
- **Avg Review Score**: The customer's average satisfaction level
- **Avg Delivery Time**: The average time taken to deliver the customer's orders

These features will make the segmentation more meaningful—capturing not just spending patterns, but also the customer experience.

In [5]:
# Customer in avg review score time

review_df = df.groupby("customer_unique_id")["review_score"].mean().reset_index()
review_df.columns = ["customer_unique_id","avg_review_score"]

# Customer in avg delivery time
df["delivery_time_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
delivery_df = df.groupby("customer_unique_id")["delivery_time_days"].mean().reset_index()
delivery_df.columns = ["customer_unique_id", "avg_delivery_Days"]

print("Review preview:")
print(review_df.head())
print("\nDelivery preview:")
print(delivery_df.head())

Review preview:
                 customer_unique_id  avg_review_score
0  0000366f3b9a7992bf8c76cfdf3221e2               5.0
1  0000b849f77a49e4a4ce2b2a4ca5be3f               4.0
2  0000f46a3911fa3c0805444483337064               3.0
3  0000f6ccb0745a6a4b88665a16c9f078               4.0
4  0004aac84e0df4da2b147fca70cf8255               5.0

Delivery preview:
                 customer_unique_id  avg_delivery_Days
0  0000366f3b9a7992bf8c76cfdf3221e2                6.0
1  0000b849f77a49e4a4ce2b2a4ca5be3f                3.0
2  0000f46a3911fa3c0805444483337064               25.0
3  0000f6ccb0745a6a4b88665a16c9f078               20.0
4  0004aac84e0df4da2b147fca70cf8255               13.0


## Step 6: Merging All Features

We now have 5 separate dataframes (recency, frequency, monetary, review, delivery); 
we will merge them all based on `customer_unique_id` to create a **final customer-level feature table**, 
which will serve as the direct input for clustering.

In [6]:
customer_features = recency_df[["customer_unique_id", "recency_days"]]
customer_features = customer_features.merge(frequency_df, on="customer_unique_id", how="left")
customer_features = customer_features.merge(monetary_df, on="customer_unique_id", how="left")
customer_features = customer_features.merge(review_df, on="customer_unique_id", how="left")
customer_features = customer_features.merge(delivery_df, on="customer_unique_id", how="left")

print("Final feature table shape:", customer_features.shape)
customer_features.head()

Final feature table shape: (93357, 6)


,customer_unique_id,recency_days,frequency,monetary,avg_review_score,avg_delivery_Days
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90,5.0,6.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19,4.0,3.0
2,0000f46a3911fa3c0805444483337064,536,1,86.22,3.0,25.0
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62,4.0,20.0
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89,5.0,13.0


In [10]:
print(customer_features.isnull().sum())

customer_unique_id      0
recency_days            0
frequency               0
monetary                0
avg_review_score      603
avg_delivery_Days       8
dtype: int64


In [12]:
# Fill the missing review score with the overall average of the dataset.

customer_features["avg_review_score"] = customer_features["avg_review_score"].fillna(
    customer_features["avg_review_score"].mean()
)

# Fill missing delivery days also with overall average
customer_features["avg_delivery_Days"] = customer_features["avg_delivery_Days"].fillna(
    customer_features["avg_delivery_Days"].mean()
)

print(customer_features.isnull().sum())


customer_unique_id    0
recency_days          0
frequency             0
monetary              0
avg_review_score      0
avg_delivery_Days     0
dtype: int64


## Step 7: Saving the Final Feature Table

This customer-level feature table (`customer_features`) is now ready for clustering.
We are saving it in `data/processed/` because it is now final, model-ready data.

In [13]:
customer_features.to_csv("../data/processed/customer_features.csv", index=False)
print("Customer features saved successfully!")
print("Final shape:", customer_features.shape)

Customer features saved successfully!
Final shape: (93357, 6)
